# MapReduce Cost Model

## Learning Objectives

By the end of this notebook you will be able to:

1. **Define** communication cost $Q$, reducer size $q$, and replication rate $r$
2. **Compute** these quantities for a given MapReduce algorithm
3. **Explain** the trade-off between $Q$ and $q$
4. **Apply** the cost model to compare alternative algorithm designs

## Why a Cost Model?

Wall-clock time for a MapReduce job is dominated by **network I/O during the shuffle phase**,
not by CPU time. The cost model quantifies this bottleneck precisely.

Three quantities capture most of what matters:

| Symbol | Name | Definition |
|--------|------|-----------|
| $Q$ | Communication cost | Total bytes in all intermediate (key, value) pairs |
| $q$ | Reducer size | Maximum bytes received by any single Reducer |
| $r$ | Replication rate | Average intermediate pairs produced per input record |

These are related: $Q \approx r \times |\text{input}|$ and $q = Q / R$ only when perfectly balanced.

## Communication Cost $Q$

$$Q = \sum_{i=1}^{M} \sum_{j=1}^{R} |\text{bytes from mapper } i \text{ to reducer } j|$$

where $M$ = number of Map tasks, $R$ = number of Reduce tasks.

### Word Count Example

- Input: $N$ words total
- Map emits one pair $(\text{word}, 1)$ per word → $Q = N$ pairs (each small, say 10 bytes) → $Q = 10N$ bytes
- This is **linear** in input size — excellent

### Similarity Join (Naive)

- Input: $N$ documents
- Naive: compare all pairs → emit $\binom{N}{2} \approx N^2/2$ pairs → $Q = O(N^2)$
- For $N = 10^6$: $Q \approx 5 \times 10^{11}$ pairs — catastrophic

The cost model reveals that the naive similarity join doesn't scale, motivating LSH.

In [ ]:
def communication_cost_word_count(n_words, bytes_per_pair=10):
    """Word count emits one pair per word."""
    Q = n_words * bytes_per_pair
    return Q


def communication_cost_naive_similarity(n_docs, bytes_per_pair=20):
    """Naive all-pairs similarity emits O(n^2) pairs."""
    Q = (n_docs * (n_docs - 1) // 2) * bytes_per_pair
    return Q


for n in [1_000, 10_000, 100_000, 1_000_000]:
    q_wc  = communication_cost_word_count(n)
    q_sim = communication_cost_naive_similarity(n)
    print(f"n={n:>10,}  word_count={q_wc:>15,} bytes  naive_sim={q_sim:>20,} bytes")

## Reducer Size $q$

$$q = \max_{j \in [R]} \,\sum_{i \in [M]} |\text{bytes from mapper } i \text{ to reducer } j|$$

$q$ measures the worst-case load on any single Reducer. A large $q$ causes **stragglers** — the
entire job waits for the slowest Reducer to finish.

### Balancing $q$

For uniform data, $q \approx Q / R$.
For skewed data (e.g., a few keys dominate), some Reducers receive far more than $Q/R$.

**Strategies to reduce $q$:**
- Increase $R$ (more Reducers, smaller load each)
- Use a custom partitioning function that accounts for key frequency
- Pre-aggregate with a Combiner to reduce intermediate output size

In [ ]:
import numpy as np
from collections import Counter


def simulate_reducer_load(word_counts, R):
    """
    Simulate how Word Count distributes load across R reducers
    using a simple hash partitioning.
    Returns: per-reducer load and the max (= q).
    """
    reducer_load = np.zeros(R, dtype=int)
    for word, count in word_counts.items():
        bucket = hash(word) % R
        reducer_load[bucket] += count   # bytes ~ proportional to count

    return reducer_load, reducer_load.max()


# Simulate a realistic word frequency distribution (Zipfian)
rng = np.random.default_rng(42)
vocab_size = 10_000
ranks = np.arange(1, vocab_size + 1)
counts = (1_000_000 / ranks).astype(int)   # Zipf: count ∝ 1/rank
words = [f"word_{i}" for i in range(vocab_size)]
word_counts = dict(zip(words, counts))
Q = sum(word_counts.values())

print(f"Total intermediate pairs Q = {Q:,}")
print()
print(f"{'R':>6}  {'q (max load)':>15}  {'Q/R (ideal)':>15}  {'imbalance':>10}")
for R in [1, 10, 100, 1000]:
    load, q = simulate_reducer_load(word_counts, R)
    ideal = Q // R
    print(f"{R:>6}  {q:>15,}  {ideal:>15,}  {q/ideal:>10.2f}x")

## Replication Rate $r$

$$r = \frac{Q}{|\text{input}|}$$

The replication rate measures how much each input record is "amplified" by the Map phase.

- $r = 1$: each record produces exactly one intermediate pair (e.g., word count, selection)
- $r > 1$: each record produces multiple pairs (e.g., join, similarity computation)
- $r < 1$: possible with heavy combiners or when most records are filtered out

### Trade-off Between $r$ and $q$

There is a fundamental trade-off: to reduce $q$ (Reducer load) you must increase $r$ (communication).

**Example — Similarity Join with groups of size $g$:**
- Partition $N$ documents into groups of $g$
- Within each group, compare all $\binom{g}{2}$ pairs → each document appears in $r$ groups
- Communication cost: $Q = r \cdot N$
- Reducer size: $q = g^2 / 2$ (comparisons per group)

As $r$ increases (more groups), $q$ decreases and vice versa.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np


N = 10_000    # number of documents
target_pairs = N * (N - 1) // 2    # all pairs we want to find

# Vary group size g and compute r and q
group_sizes = np.arange(2, N + 1, 100)
r_vals = []
q_vals = []

for g in group_sizes:
    # Number of groups needed to cover all pairs (each doc in r groups)
    # For g documents per group: r = N/g groups, each doc replicated to ~N/g groups
    # q = g*(g-1)/2  (pairs compared per reducer)
    n_groups = N // g
    r = N / n_groups if n_groups > 0 else N
    q = g * (g - 1) / 2
    r_vals.append(r)
    q_vals.append(q)

fig, ax = plt.subplots(figsize=(7, 4))
ax2 = ax.twinx()

ax.plot(group_sizes, r_vals, color="steelblue", label="replication rate r")
ax2.plot(group_sizes, q_vals, color="tomato", label="reducer size q")

ax.set_xlabel("group size g")
ax.set_ylabel("replication rate r", color="steelblue")
ax2.set_ylabel("reducer size q (pairs per reducer)", color="tomato")
ax.set_title("r vs q trade-off for Similarity Join")
ax.legend(loc="upper left"); ax2.legend(loc="upper right")
plt.tight_layout()
plt.savefig("/tmp/mr_cost_tradeoff.png", dpi=100)
plt.close()
print("saved /tmp/mr_cost_tradeoff.png")
print()
print(f"{'g':>8}  {'r':>12}  {'q':>15}")
for g, r, q in list(zip(group_sizes, r_vals, q_vals))[::10]:
    print(f"{g:>8}  {r:>12.1f}  {q:>15.0f}")

## Summary

| Quantity | Formula | Goal | Danger |
|----------|---------|------|--------|
| Communication cost $Q$ | $\sum_{i,j} \text{bytes}_{ij}$ | Minimise | $O(N^2)$ algorithms at scale |
| Reducer size $q$ | $\max_j \sum_i \text{bytes}_{ij}$ | Minimise | Stragglers, OOM on large keys |
| Replication rate $r$ | $Q / \|\text{input}\|$ | Low as possible | Amplifies with joins and similarity |

**Design principle:** when $q$ is fixed by memory constraints, choose the algorithm that
minimises $Q$ subject to that constraint. This often means finding the right grouping
or partitioning strategy.